# Reusable Template – Binary Logistic Loss & Cost

**Short name:** `Logistic_Loss_Cost`  
Copy this notebook for any new 0/1 problem where you need to *score* a candidate $(w,b)$ before (or instead of) running full gradient descent.

Replace the data-loading cell, then run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

def bce_loss(f, y):
    f = np.clip(np.asarray(f, dtype=float), 1e-15, 1 - 1e-15)
    y = np.asarray(y, dtype=float)
    return -(y * np.log(f) + (1 - y) * np.log(1 - f))

def compute_cost(X, y, w, b):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    z = (w * X + b) if X.ndim == 1 else (X @ w + b)
    return float(np.mean(bce_loss(sigmoid(z), y)))

def predict_proba(X, w, b):
    X = np.asarray(X, dtype=float)
    w = np.asarray(w, dtype=float)
    z = (w * X + b) if X.ndim == 1 else (X @ w + b)
    return sigmoid(z)



## 1. Load data — replace this cell


In [ ]:
# Example: 2-D lab file shipped with this project
arr = np.loadtxt("data/logistic_loss_2d.csv", delimiter=",", skiprows=1)
X, y = arr[:, :2], arr[:, 2]
print(X.shape, y.shape, "positive rate", y.mean())



## 2. Score one or more candidate parameters


In [ ]:
candidates = [
    ("lab b=-3", np.array([1.0, 1.0]), -3.0),
    ("lab b=-4", np.array([1.0, 1.0]), -4.0),
    ("baseline 0", np.zeros(X.shape[1]), 0.0),
]
for name, w, b in candidates:
    print(f"{name:16s}  J={compute_cost(X, y, w, b):.6f}")



## 3. Optional: 1-D / 2-D sketch


In [ ]:
if X.shape[1] == 2:
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    ax.scatter(X[y == 0, 0], X[y == 0, 1], label="y=0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], marker="x", label="y=1")
    ax.legend(); ax.set_title("features")
    plt.show()



## 4. Mini simulation — label noise vs cost of a frozen model


In [ ]:
w_frozen, b_frozen = np.array([1.0, 1.0]), -3.0
rng = np.random.default_rng(0)
rates, js = [], []
for noise in np.linspace(0, 0.4, 9):
    y_noisy = y.copy()
    nflip = int(noise * len(y))
    if nflip:
        idx = rng.choice(len(y), size=nflip, replace=False)
        y_noisy[idx] = 1 - y_noisy[idx]
    rates.append(noise)
    js.append(compute_cost(X, y_noisy, w_frozen, b_frozen))
plt.plot(rates, js, "o-")
plt.xlabel("flip rate"); plt.ylabel("J"); plt.title("Frozen model vs noisy labels")
plt.grid(True, alpha=0.3); plt.show()

